# MVT-Flow Training (voraus-AD) — Colab GPU, self-contained

No repo clone needed. Set **Runtime → GPU**, then **Runtime → Run all**.

Faithful to Brockmann et al. (2023), arXiv:2311.04765 (4 coupling blocks, kernels
[13,1,1], dilations [2,1,1], Adam 8e-4 decayed at epochs 11/61, 70 epochs, batch 32).
Builds **correct signal-major (130, 1100)** windows — the earlier notebook collapsed
the time axis (`n_timesteps=1`) and scrambled the signal/timestep layout.
> **Colab-GPU counterpart of `scripts/train_mvtflow.py`.** Same logic, run on a free
> GPU because normalizing-flow training is the one compute-heavy step. The `scripts/`
> version is the canonical/local trainer — this notebook is not a second source of truth.

### Where each part lives in the repo

| Notebook section | Shipped file |
|---|---|
| 3. MVT-Flow model (coupling blocks, flow, detector) | `src/mvt_flow_model.py` |
| 4. Preprocess -> train -> evaluate -> save | `scripts/train_mvtflow.py` |
| Threshold calibration (Tukey fences) | `scripts/calibrate_detection.py` |
| Inference on live sensor windows | `src/detection.py` |
| Decision from the detection status | `src/fusion.py` |


## 1. Dependencies

In [ ]:
!pip install -q torch pandas pyarrow scikit-learn

## 2. Get the dataset (voraus-AD 100 Hz parquet, ~1 GB)

In [ ]:
PARQUET = 'voraus-ad-dataset-100hz.parquet'
import os
# Option A - download from your Google Drive by file id (share: 'Anyone with the link'):
FILE_ID = '1DjzTx7N9Kt7fBnUPdkzt90GpqTziw31K'   # replace with yours
if not os.path.exists(PARQUET):
    !pip install -q gdown && gdown --id $FILE_ID -O $PARQUET
# Option B - mount Drive and copy instead:
# from google.colab import drive; drive.mount('/content/drive')
# !cp '/content/drive/MyDrive/Predictive-Maintenance/'$PARQUET $PARQUET


## 3. MVT-Flow model

In [ ]:
"""
MVT-Flow Model for voraus-AD Anomaly Detection

This module provides the MVT-Flow (Multivariate Time-series Flow) implementation
for anomaly detection on robot sensor data.

Based on: Brockmann et al. (2023) - The voraus-AD Dataset for Anomaly Detection
in Robot Applications (arXiv:2311.04765)

Architecture:
- Real-NVP normalizing flow with 4 coupling blocks
- 1D convolutional internal networks for temporal modeling
- Soft-clamping for training stability
- Maximum likelihood training on normal data

Author: Luca Zagaia
"""

import torch
import torch.nn as nn
import numpy as np
from typing import Tuple, List, Optional
import pickle
from pathlib import Path


class CouplingBlock(nn.Module):
    """Real-NVP coupling block for multivariate time series."""
    
    def __init__(
        self,
        n_signals: int,
        n_timesteps: int,
        hidden_channels: int = 2,
        kernel_sizes: List[int] = [13, 1, 1],
        dilations: List[int] = [2, 1, 1],
        alpha: float = 1.9
    ):
        super().__init__()
        
        self.n_signals = n_signals
        self.n_timesteps = n_timesteps
        self.alpha = alpha
        
        perm = torch.randperm(n_signals)
        self.register_buffer('permutation', perm)
        
        self.split_size = n_signals // 2
        
        self.g1 = self._build_conv_net(
            self.split_size, 
            self.split_size * hidden_channels,
            kernel_sizes, 
            dilations
        )
        
        self.g2 = self._build_conv_net(
            self.split_size,
            self.split_size * hidden_channels,
            kernel_sizes,
            dilations
        )
        
    def _build_conv_net(
        self,
        in_channels: int,
        hidden_channels: int,
        kernel_sizes: List[int],
        dilations: List[int]
    ) -> nn.Sequential:
        """Build 1D convolutional network."""
        layers = []
        
        layers.extend([
            nn.Conv1d(
                in_channels,
                hidden_channels,
                kernel_size=kernel_sizes[0],
                padding=(kernel_sizes[0] - 1) * dilations[0] // 2,
                dilation=dilations[0]
            ),
            nn.ReLU()
        ])
        
        for i in range(1, len(kernel_sizes) - 1):
            layers.extend([
                nn.Conv1d(
                    hidden_channels,
                    hidden_channels,
                    kernel_size=kernel_sizes[i],
                    padding=(kernel_sizes[i] - 1) * dilations[i] // 2,
                    dilation=dilations[i]
                ),
                nn.ReLU()
            ])
        
        layers.append(
            nn.Conv1d(
                hidden_channels,
                in_channels * 2,
                kernel_size=kernel_sizes[-1],
                padding=(kernel_sizes[-1] - 1) * dilations[-1] // 2,
                dilation=dilations[-1]
            )
        )
        
        return nn.Sequential(*layers)
    
    def _soft_clamp(self, s: torch.Tensor) -> torch.Tensor:
        """Soft-clamping to prevent exploding gradients."""
        return (2 * self.alpha / np.pi) * torch.atan(s * np.pi / (2 * self.alpha))
    
    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Forward pass: x -> z"""
        x = x[:, self.permutation, :]
        x1, x2 = torch.split(x, self.split_size, dim=1)
        
        out1 = self.g1(x1)
        s1, t1 = torch.chunk(out1, 2, dim=1)
        s1 = self._soft_clamp(s1)
        y2 = x2 * torch.exp(s1) + t1
        
        out2 = self.g2(y2)
        s2, t2 = torch.chunk(out2, 2, dim=1)
        s2 = self._soft_clamp(s2)
        y1 = x1 * torch.exp(s2) + t2
        
        y = torch.cat([y1, y2], dim=1)
        log_det_jacobian = torch.sum(s1, dim=[1, 2]) + torch.sum(s2, dim=[1, 2])
        
        return y, log_det_jacobian
    
    def inverse(self, y: torch.Tensor) -> torch.Tensor:
        """Inverse pass: z -> x"""
        y1, y2 = torch.split(y, self.split_size, dim=1)
        
        out2 = self.g2(y2)
        s2, t2 = torch.chunk(out2, 2, dim=1)
        s2 = self._soft_clamp(s2)
        x1 = (y1 - t2) * torch.exp(-s2)
        
        out1 = self.g1(x1)
        s1, t1 = torch.chunk(out1, 2, dim=1)
        s1 = self._soft_clamp(s1)
        x2 = (y2 - t1) * torch.exp(-s1)
        
        x = torch.cat([x1, x2], dim=1)
        
        inv_permutation = torch.argsort(self.permutation)
        x = x[:, inv_permutation, :]
        
        return x


class MVTFlow(nn.Module):
    """MVT-Flow: Multivariate Time-series Normalizing Flow"""
    
    def __init__(
        self,
        n_signals: int,
        n_timesteps: int,
        n_blocks: int = 4,
        hidden_channels: int = 2,
        kernel_sizes: List[int] = [13, 1, 1],
        dilations: List[int] = [2, 1, 1],
        alpha: float = 1.9
    ):
        super().__init__()
        
        self.n_signals = n_signals
        self.n_timesteps = n_timesteps
        self.n_blocks = n_blocks
        
        self.blocks = nn.ModuleList([
            CouplingBlock(
                n_signals,
                n_timesteps,
                hidden_channels,
                kernel_sizes,
                dilations,
                alpha
            ) for _ in range(n_blocks)
        ])
    
    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Forward pass: data -> latent space"""
        z = x
        log_det_jacobian = 0
        
        for block in self.blocks:
            z, ldj = block(z)
            log_det_jacobian += ldj
        
        return z, log_det_jacobian
    
    def inverse(self, z: torch.Tensor) -> torch.Tensor:
        """Inverse pass: latent space -> data"""
        x = z
        for block in reversed(self.blocks):
            x = block.inverse(x)
        return x
    
    def log_prob(self, x: torch.Tensor) -> torch.Tensor:
        """Compute log probability of data under the model."""
        z, log_det_jacobian = self.forward(x)
        log_pz = -0.5 * torch.sum(z ** 2, dim=[1, 2])
        log_px = log_pz + log_det_jacobian
        return log_px
    
    def compute_loss(self, x: torch.Tensor) -> torch.Tensor:
        """Compute negative log-likelihood loss."""
        log_prob = self.log_prob(x)
        return -torch.mean(log_prob)
    
    def anomaly_score(self, x: torch.Tensor) -> torch.Tensor:
        """Compute anomaly score (negative log probability)."""
        with torch.no_grad():
            log_prob = self.log_prob(x)
        return -log_prob


class MVTFlowDetector:
    """
    Production-ready wrapper for MVT-Flow anomaly detection.
    
    This class provides a simplified interface for using MVT-Flow in the
    detection pipeline, handling model loading, preprocessing, and inference.
    """
    
    def __init__(
        self,
        model_path: str,
        scaler_path: str,
        n_signals: int = 130,
        n_timesteps: int = 1100,
        device: str = None
    ):
        """
        Initialize MVT-Flow detector.
        
        Args:
            model_path: Path to trained .pt model file
            scaler_path: Path to fitted StandardScaler .pkl file
            n_signals: Number of sensor signals (default: 130 for voraus-AD)
            n_timesteps: Length of time series windows (default: 1100)
            device: Device to run on ('cuda', 'cpu', or None for auto-detect)
        """
        self.n_signals = n_signals
        self.n_timesteps = n_timesteps
        
        # Device selection
        if device is None:
            self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        else:
            self.device = device
        
        # Load model
        self.model = MVTFlow(
            n_signals=n_signals,
            n_timesteps=n_timesteps,
            n_blocks=4,
            hidden_channels=2,
            kernel_sizes=[13, 1, 1],
            dilations=[2, 1, 1],
            alpha=1.9
        )
        
        # Load trained weights
        checkpoint = torch.load(model_path, map_location=self.device)
        if 'model_state_dict' in checkpoint:
            self.model.load_state_dict(checkpoint['model_state_dict'])
        else:
            self.model.load_state_dict(checkpoint)
        
        self.model.to(self.device)
        self.model.eval()
        
        # Load scaler
        with open(scaler_path, 'rb') as f:
            self.scaler = pickle.load(f)
        
        print(f"✅ MVT-Flow detector loaded successfully")
        print(f"   Device: {self.device}")
        print(f"   Signals: {n_signals}, Timesteps: {n_timesteps}")
    
    def preprocess(self, X: np.ndarray) -> torch.Tensor:
        """
        Preprocess raw sensor data for MVT-Flow.
        
        Args:
            X: Raw sensor data of shape (n_samples, n_signals, n_timesteps)
        
        Returns:
            Preprocessed tensor ready for model input
        """
        n_samples, n_signals, n_timesteps = X.shape
        
        # Reshape for standardization
        X_flat = X.transpose(0, 2, 1).reshape(-1, n_signals)
        
        # Standardize
        X_scaled = self.scaler.transform(X_flat)
        
        # Reshape back
        X_scaled = X_scaled.reshape(n_samples, n_timesteps, n_signals).transpose(0, 2, 1)
        
        # Convert to tensor
        X_tensor = torch.FloatTensor(X_scaled).to(self.device)
        
        return X_tensor
    
    def predict_anomaly_score(self, X: np.ndarray) -> np.ndarray:
        """
        Compute anomaly scores for sensor data.
        
        Args:
            X: Raw sensor data of shape (n_samples, n_signals, n_timesteps)
        
        Returns:
            Anomaly scores (higher = more anomalous)
        """
        X_tensor = self.preprocess(X)
        
        with torch.no_grad():
            scores = self.model.anomaly_score(X_tensor)
        
        return scores.cpu().numpy()
    
    def predict_anomaly_probability(
        self, 
        X: np.ndarray,
        normal_threshold: float = None
    ) -> np.ndarray:
        """
        Convert anomaly scores to probabilities [0, 1].
        
        Args:
            X: Raw sensor data
            normal_threshold: Anomaly score threshold for normal data
                            (typically computed from validation set)
        
        Returns:
            Anomaly probabilities [0, 1]
        """
        scores = self.predict_anomaly_score(X)
        
        if normal_threshold is None:
            # Use simple sigmoid transformation
            # Map scores to [0, 1] with reasonable defaults
            mean_score = scores.mean()
            std_score = scores.std()
            
            # Normalize to z-scores, then sigmoid
            z_scores = (scores - mean_score) / (std_score + 1e-8)
            probabilities = 1 / (1 + np.exp(-z_scores))
        else:
            # Use threshold-based approach
            # Scores above threshold map to high probability
            excess = scores - normal_threshold
            probabilities = 1 / (1 + np.exp(-excess / 1000))  # Scale factor for smooth transition
        
        return probabilities
    
    def predict(
        self,
        X: np.ndarray,
        threshold: float = 0.5
    ) -> np.ndarray:
        """
        Binary anomaly predictions.
        
        Args:
            X: Raw sensor data
            threshold: Probability threshold for anomaly classification
        
        Returns:
            Binary predictions (0=normal, 1=anomaly)
        """
        probabilities = self.predict_anomaly_probability(X)
        return (probabilities >= threshold).astype(int)


## 4. Preprocess → train → evaluate → save

In [ ]:
import numpy as np, pandas as pd, pyarrow.parquet as pq, torch, pickle, json, os
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

META = ['time','sample','anomaly','category','setting','action','active']
N_SIGNALS, WINDOW = 130, 1100

# --- build correct signal-major (130, 1100) windows, one per recording ---
cols = pq.ParquetFile(PARQUET).schema_arrow.names
SIG = [c for c in cols if c not in META]; assert len(SIG) == N_SIGNALS, len(SIG)
tbl = pq.read_table(PARQUET, columns=SIG + ['sample','anomaly'])
sig = np.column_stack([tbl.column(c).to_numpy(zero_copy_only=False) for c in SIG]).astype('float32')
sample = tbl.column('sample').to_numpy(); anom = tbl.column('anomaly').to_numpy()
idx = pd.Series(np.arange(len(sample))).groupby(sample).apply(lambda s: s.values).to_dict()
label = pd.Series(anom).groupby(sample).first()
def window(sid):
    w = sig[idx[sid]]
    return (w[:WINDOW] if len(w) >= WINDOW else np.pad(w, ((0, WINDOW-len(w)), (0,0)), mode='edge')).T
normal  = np.stack([window(s) for s in label[~label].index])
anomaly = np.stack([window(s) for s in label[ label].index])
print('windows: normal', normal.shape, 'anomaly', anomaly.shape)

# --- semi-supervised: train on NORMAL only; hold out normal + anomaly for AUROC ---
rng = np.random.default_rng(42); perm = rng.permutation(len(normal)); nt = int(0.3*len(normal))
test_n, train_n = normal[perm[:nt]], normal[perm[nt:]]
scaler = StandardScaler().fit(train_n.transpose(0,2,1).reshape(-1, N_SIGNALS))
def scale(X):
    n,s,t = X.shape
    return scaler.transform(X.transpose(0,2,1).reshape(-1,s)).reshape(n,t,s).transpose(0,2,1).astype('float32')
train_s, testn_s, testa_s = scale(train_n), scale(test_n), scale(anomaly)

# --- train MVT-Flow with the paper's hyperparameters ---
device = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device:', device)
torch.manual_seed(42)
model = MVTFlow(n_signals=N_SIGNALS, n_timesteps=WINDOW, n_blocks=4, hidden_channels=2,
                kernel_sizes=[13,1,1], dilations=[2,1,1], alpha=1.9).to(device)
opt = torch.optim.Adam(model.parameters(), lr=8e-4)
sched = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=[11,61], gamma=0.1)
loader = DataLoader(TensorDataset(torch.from_numpy(train_s)), batch_size=32, shuffle=True)
for epoch in range(70):
    model.train(); run = 0.0
    for (xb,) in loader:
        xb = xb.to(device); loss = model.compute_loss(xb)
        opt.zero_grad(); loss.backward(); opt.step(); run += loss.item()*len(xb)
    sched.step()
    if epoch % 5 == 0 or epoch == 69: print(f'  epoch {epoch:2d}  nll={run/len(train_n):.2f}')

# --- AUROC + calibrated thresholds ---
model.eval()
def scores(X):
    out = []
    with torch.no_grad():
        for i in range(0, len(X), 32):
            out.append(model.anomaly_score(torch.from_numpy(X[i:i+32]).to(device)).cpu().numpy())
    return np.concatenate(out)
sn, sa = scores(testn_s), scores(testa_s)
auroc = roc_auc_score(np.r_[np.zeros(len(sn)), np.ones(len(sa))], np.r_[sn, sa])
warn, anomT = float(np.percentile(sn, 95)), float(np.percentile(sn, 99))
print(f'\nAUROC = {auroc:.3f}   (paper baseline ~0.936)')
print(f'thresholds: warning(p95)={warn:.2f}  anomaly(p99)={anomT:.2f}')

os.makedirs('models', exist_ok=True)
torch.save(model.state_dict(), 'models/mvt_flow_voraus_ad.pt')
pickle.dump(scaler, open('models/scaler_voraus_ad.pkl','wb'))
json.dump({'warning_threshold': warn, 'anomaly_threshold': anomT, 'auroc': round(float(auroc),4),
           'calibrated_on': f'{len(test_n)} held-out normal windows (p95/p99)'},
          open('models/voraus_thresholds.json','w'), indent=2)


## 5. Keep the weights (Colab wipes local files at session end)

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os, shutil
DEST = '/content/drive/MyDrive/Predictive-Maintenance/models'; os.makedirs(DEST, exist_ok=True)
for f in ['mvt_flow_voraus_ad.pt','scaler_voraus_ad.pkl','voraus_thresholds.json']:
    shutil.copy('models/'+f, DEST)
